In [1]:
%load_ext autoreload
%autoreload 2
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'
import time
import torch
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

from nnsight import  NNsight, LanguageModel
from transformers import AutoTokenizer, AutoConfig, AutoModelForCausalLM
from datasets import load_dataset

In [2]:
SPLIT = "train"
NUM_EXAMPLES = 100

TASKS_TO_HF_NAMES = {
    'ioi': 'ioi',
    'mcqa': 'copycolors_mcqa',
    'arithmetic_addition': 'arithmetic_addition',
    'arithmetic_subtraction': 'arithmetic_subtraction',
    'arc_easy': 'arc_easy',
    'arc_challenge': 'arc_challenge',
}

In [3]:
class MIBDataset(Dataset):
    """Minimal MIB dataset loader."""

    def __init__(self, task, tokenizer, model_name, split='train', num_examples=100):
        self.task = task
        self.tokenizer = tokenizer
        self.model_name = model_name

        hf_url = f"mib-bench/{TASKS_TO_HF_NAMES[task]}"
        if task == 'mcqa':
            self.dataset = load_dataset(hf_url, '4_answer_choices', split=split)
            self.counterfactual_type = "symbol_counterfactual"
        elif task.startswith('arc'):
            self.dataset = load_dataset(hf_url, split=split)
            self.counterfactual_type = "symbol_counterfactual"
        elif task.startswith('arithmetic'):
            self.dataset = load_dataset(hf_url, split=split)
            self.operator = "-" if "subtraction" in task else "+"
        else:
            self.dataset = load_dataset(hf_url, split=split)

        self.dataset = self._filter()
        if num_examples and num_examples < len(self.dataset):
            self.dataset = self.dataset.select(range(num_examples))

    def _filter(self):
        tok = self.tokenizer
        if self.task == 'ioi':
            return self.dataset.filter(
                lambda x: (
                    len(tok(f" {x['metadata']['indirect_object']}", add_special_tokens=False).input_ids) ==
                    len(tok(f" {x['metadata']['subject']}", add_special_tokens=False).input_ids) and
                    len(tok(f" {x['metadata']['indirect_object']}", add_special_tokens=False).input_ids) ==
                    len(tok(f" {x['metadata']['random_c']}", add_special_tokens=False).input_ids)
                )
            )
        elif self.task == 'mcqa' or self.task.startswith('arc'):
            ct = self.counterfactual_type
            return self.dataset.filter(
                lambda x: (
                    len(tok(x["choices"]["label"][x["answerKey"]], add_special_tokens=False).input_ids) ==
                    len(tok(str(x[ct]["choices"]["label"][x[ct]["answerKey"]]), add_special_tokens=False).input_ids)
                )
            )
        elif self.task.startswith('arithmetic'):
            op = self.operator
            return self.dataset.filter(
                lambda x: (
                    len(tok(str(x["label"]), add_special_tokens=False).input_ids) == 1 and
                    x["random_counterfactual"] is not None and
                    x["random_counterfactual"]["prompt"] is not None and
                    x["operator"] == op and
                    len(tok(str(x["random_counterfactual"]["label"]), add_special_tokens=False).input_ids) == 1
                )
            )
        return self.dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        row = self.dataset[index]
        tok = self.tokenizer

        if self.task == 'ioi':
            correct_idx = tok(f" {row['metadata']['indirect_object']}", add_special_tokens=False).input_ids[0]
            incorrect_idx = tok(f" {row['metadata']['subject']}", add_special_tokens=False).input_ids[0]
            cf = row.get("s2_io_flip_counterfactual", row.get("counterfactual", {}))
            return row["prompt"], cf.get("prompt", row["prompt"]), [correct_idx, incorrect_idx]

        elif self.task == 'mcqa' or self.task.startswith('arc'):
            ct = self.counterfactual_type
            correct_idx = tok(row["choices"]["label"][row["answerKey"]], add_special_tokens=False).input_ids[0]
            cf = row[ct]
            incorrect_idx = tok(str(cf["choices"]["label"][cf["answerKey"]]), add_special_tokens=False).input_ids[0]
            return row["prompt"], cf["prompt"], [correct_idx, incorrect_idx]

        elif self.task.startswith('arithmetic'):
            correct_idx = tok(str(row["label"]), add_special_tokens=False).input_ids[0]
            cf = row["random_counterfactual"]
            incorrect_idx = tok(str(cf["label"]), add_special_tokens=False).input_ids[0]
            return row["prompt"], cf["prompt"], [correct_idx, incorrect_idx]
        
    def dataloader(self, batch_size: int) -> DataLoader:
        return DataLoader(
            self, batch_size=batch_size,
            collate_fn=self._collate_fn, shuffle=False,
        )

    @staticmethod
    def _collate_fn(xs):
        clean, corrupted, labels = zip(*xs)
        clean_labels, corrupt_labes = zip(*labels)
        return list(clean), list(corrupted), clean_labels, corrupt_labes 

In [11]:
model_id = "meta-llama/Llama-3.1-8B"
model_id = 'Qwen/Qwen2.5-0.5B'
model_id = 'google/gemma-2-2b'
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.padding_side = 'left'
if not tokenizer.pad_token:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.bfloat16, attn_implementation="eager", device_map='auto').eval()
# model = NNsight(model)
from linear_transformer import patch_model_for_lvp
from linear_transformer.modules import sec_jac_softmax, constant_softmax, integrated_softmax
model = patch_model_for_lvp(model, attn_act_fn=sec_jac_softmax, nnsight_wrapper=False)
model = NNsight(model)

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

In [5]:
batch_size = 32
dataset = MIBDataset('ioi', tokenizer, model_id, SPLIT)
dataloader = dataset.dataloader(batch_size)

In [12]:
folder_path = os.getcwd()
import sys
if folder_path not in sys.path:
    sys.path.insert(0, folder_path)

from tracer.tracer import EdgeCircuitTracer
from tracer.model_adapters import Llama2ModelAdapter, Gemma2ModelAdapter, FrozenGemma2ModelAdapter

# adapter = Llama2ModelAdapter(model)
adapter = FrozenGemma2ModelAdapter(model)
tracer = EdgeCircuitTracer(adapter, tokenizer)

circuit = tracer.build_circuit(dataloader, use_counterfactual=False) # , use_counterfactual=True

100%|██████████| 4/4 [00:01<00:00,  3.13it/s]


In [13]:
# embedding (src): input

# attention src: a{layer_id}.h{head_id} e.g. a0.h21
# attention tgt: a{layer_id}.h{head_id}<{head_type}> e.g. a0.h21<k>

# mlp src / tgt: m{layer_id} e.g. m2

# lm_head (tgt): logit

# edge: {src}->{tgt} : {'score': {score}, 'in_graph': False}

# Json Schema
# {
#   "cfg": {
#     "n_layers": 32,
#     "n_heads": 32,
#     "parallel_attn_mlp": False,
#     "d_model": 4096
#   },
#   "nodes": {},
#   "edges": {},
# }

def get_src_id(name: str, n_heads):
    if name == 'input':
        return 0
    if name[0] == 'm':
        layer_id = int(name[1:])
        return 1 + layer_id * (n_heads + 1) + n_heads
    if name[0] == 'a':
        attn_mod, head = name.split('.')
        layer_id = int(attn_mod[1:])
        head_id = int(head[1:])
        return 1 + layer_id * (n_heads + 1) + head_id
    raise NotImplementedError(f"Name '{name}' does not fit the patterns.")

def get_tgt_id(name: str, n_heads):
    if name == 'logits':
        return -1
    if name[0] == 'm':
        layer_id = int(name[1:])
        return layer_id * (3 * n_heads + 1) + 3 * n_heads
    if name[0] == 'a':
        attn_mod, head = name.split('.')
        layer_id = int(attn_mod[1:])
        head_id = int(head[1:-3])
        head_type = head[-2]
        if head_type == 'q':
            return layer_id * (3 * n_heads + 1) + head_id
        if head_type == 'k':
            return layer_id * (3 * n_heads + 1) + n_heads + head_id
        if head_type == 'v':
            return layer_id * (3 * n_heads + 1) + 2 * n_heads + head_id
    raise NotImplementedError(f"Name '{name}' does not fit the patterns.")

def create_mib_circuit(scores: torch.Tensor, n_layers, n_heads, d_model, circuit_level: str = 'edge'):
    circuit = {  
        "cfg": {
            "n_layers": n_layers,
            "n_heads": n_heads,
            "parallel_attn_mlp": False,
            "d_model": d_model
        },
        'nodes': {},
        'edges': {}
    }


    def _inner_loop(src_name: str, src_layer_id: int = 0, src_head_id: int = None):
        src_id = get_src_id(src_name, n_heads)
        for tgt_layer_id in range(src_layer_id, n_layers):

            if tgt_layer_id > src_layer_id or src_name == 'input':
                for tgt_head_type in ['q', 'k', 'v']:
                    for tgt_head_id in range(n_heads):
                        tgt_name = f"a{tgt_layer_id}.h{tgt_head_id}<{tgt_head_type}>"
                        tgt_id = get_tgt_id(tgt_name, n_heads)
                        circuit['edges'][f"{src_name}->{tgt_name}"] = {'score': scores[src_id, tgt_id].item(), 'in_graph': False}

            if tgt_layer_id > src_layer_id and not src_name.startswith('m'):
                tgt_name = f"m{tgt_layer_id}"
                tgt_id = get_tgt_id(tgt_name, n_heads)
                circuit['edges'][f"{src_name}->{tgt_name}"] = {'score': scores[src_id, tgt_id].item(), 'in_graph': False}
        
        circuit['edges'][f"{src_name}->logits"] = {'score': scores[src_id, -1].item(), 'in_graph': False}

    
    # outer loop
    _inner_loop("input") 
    circuit["nodes"]["input"] = {'in_graph': False}

    for layer_id in range(n_layers):
        for head_id in range(n_heads):
            src_name = f"a{layer_id}.h{head_id}"
            _inner_loop(src_name, src_layer_id=layer_id, src_head_id=head_id)
            circuit["nodes"][src_name] = {'in_graph': False}
        src_name = f"m{layer_id}"
        _inner_loop(src_name, src_layer_id=layer_id)
        circuit["nodes"][src_name] = {'in_graph': False}


    return circuit

In [14]:
L = adapter.n_layers
H = adapter.n_heads
D = adapter.model_dim
circuit = create_mib_circuit(circuit, L, H, D)
import json
with open('circuits/test3_importance.json', 'w') as f:
    json.dump(circuit, f, indent=2)

In [15]:
s = slice(-1,)
[1, 2, 3][s]

[1, 2]

In [16]:
"""
CUDA_VISIBLE_DEVICES=1 python experiments/mib/MIB-circuit-track/run_evaluation.py --models gemma2 --tasks ioi --split test --batch-size 8 --head 200 --circuit-files /home/dacslab/lasse_jantsch/circuit_discovery/circuits/test3_importance.json --output-dir /home/dacslab/lasse_jantsch/circuit_discovery/experiments/mib/results

python experiments/mib/MIB-circuit-track/print_results.py --output-dir /home/dacslab/lasse_jantsch/circuit_discovery/experiments/mib/results --split test --metric cpr
"""

'\nCUDA_VISIBLE_DEVICES=1 python experiments/mib/MIB-circuit-track/run_evaluation.py --models gemma2 --tasks ioi --split test --batch-size 8 --head 200 --circuit-files /home/dacslab/lasse_jantsch/circuit_discovery/circuits/test3_importance.json --output-dir /home/dacslab/lasse_jantsch/circuit_discovery/experiments/mib/results\n\npython experiments/mib/MIB-circuit-track/print_results.py --output-dir /home/dacslab/lasse_jantsch/circuit_discovery/experiments/mib/results --split test --metric cpr\n'